In [1]:
# Внимание!!! Важно, что бы файлы с данными и исполняемый файл находились в одной папке, 
# тогда пути к тестовым и тренировочным наборам будут содержать только имена файлов.
# 
# В пути к тренировочным и тестовым данным запрежается использовать абсалютную адресацию, 
# то есть адресацию, в которой присутствуют имена папок. Путь должен содержать только имя файла.
#
# Напоминание: под моделью машинного обучения понимаются все действия с исходными данными, 
# которые необходимо произвести, что бы сопоставить признаки целевому значению.

### Область работы 1 (библиотеки)

In [2]:
# Данный блок в области 1 выполняется преподавателем
# 
# данный блок предназначен только для подключения необходимых библиотек
# запрещается подключать библиотеки в других блоках
# запрещается скрывать предупреждения системы
# установка дополнительных библиотек размещается прямо здесь (обязательно закоментированы)
# pip install

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_log_error
from IPython.display import display
from sklearn.datasets import load_iris
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.preprocessing import RobustScaler, Normalizer
from sklearn.preprocessing import PolynomialFeatures, OneHotEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.model_selection import GridSearchCV, KFold, StratifiedShuffleSplit
from sklearn.model_selection import ParameterGrid
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import QuantileTransformer, PowerTransformer, FunctionTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.svm import SVR
from sklearn.linear_model import Ridge, Lasso, ElasticNet, Lars, OrthogonalMatchingPursuit, SGDRegressor
from sklearn.metrics import classification_report
import category_encoders as ce
from sklearn.impute import SimpleImputer
import sklearn

print(sklearn.__version__)


1.7.2


### Область работы 2 (выполнение лучшей модели)

In [4]:
# Данный блок(и) в области 2 выполняется преподавателем
#
# В области находится одна, единственная, итоговая модель машинного обучения с однозначными, 
# зафиксированными параметрами
#
# В данной области категорически запрещается искать, выбирать, улучшать, оптимизировать, 
# тюниговать и т.д. модель машинного обучения

In [5]:
# Путь к тренировочному набору
path_train = 'train.csv' # содержит только имя файла, без имен папок
# Путь к тестовому набору
path_test  = 'test.csv' # содержит только имя файла, без имен папок

In [6]:
# Блок(и) обучения и поверки модели

In [7]:
df_train = pd.read_csv(path_train)
df_test = pd.read_csv(path_test)

In [8]:
y = np.array(df_train.total_sales_price)
X = df_train.drop(columns=['total_sales_price'])

X_test = df_test

In [9]:
num_features = ['size','depth_percent','table_percent']
meas_features = ['meas_length', 'meas_width', 'meas_depth']
girdle_features = ['girdle_min', 'girdle_max']
b_ordered_features = ['cut', 'symmetry', 'polish']
ordered_features = ['color', 'clarity', 'fluor_intensity', 'eye_clean'] 
one_hot_features = ['fluor_color']

In [10]:
fluor_color_transformer = Pipeline(steps=[
    ('missing_сat', SimpleImputer(strategy = 'most_frequent')),
    ("cat_encoding", OneHotEncoder(sparse_output=False))])

_girdle_map = {
    'XTN': 0,   
    'VTN': 1,   
    'STN': 2,   
    'TN' : 3,  
    'M'  : 4,  
    'STK': 5,   
    'TK' : 6,   
    'VTK': 7,   
    'XTK': 8   
}

girdle_both_map = [
    {'col': 'girdle_min', 'mapping': _girdle_map},
    {'col': 'girdle_max', 'mapping': _girdle_map},
]
girdle_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ord', ce.OrdinalEncoder(mapping=girdle_both_map))
])

num_transformer = Pipeline(steps=[
    ('missing_num', SimpleImputer(strategy = 'mean')),
    ('scaler', MinMaxScaler())])

meas_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('polynom', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler())
])

binary_transformer = Pipeline(steps=[
    ('missing_num', SimpleImputer(strategy='most_frequent')),
    ('ce', ce.OrdinalEncoder(cols=b_ordered_features))   
])

color_map = [{
    'col': 'color',
    'mapping': {
        'D': 10, 'E': 9, 'F': 8, 'G': 7, 'H': 6,
        'I': 5, 'J': 4, 'K': 3, 'L': 2, 'M': 1
    }
}]
color_transformer = Pipeline(steps=[
    ('missing_num', SimpleImputer(strategy='most_frequent')),
    ('ce', ce.OrdinalEncoder(mapping=color_map))
])


clarity_map = [{
    'col': 'clarity',
    'mapping': {
        'FL': 10, 'IF': 9, 'VVS1': 8, 'VVS2': 7, 'VS1': 6, 'VS2': 5,
        'SI1': 4, 'SI2': 3, 'I1': 2, 'I2': 1, 'I3': 0
    }
}]
clarity_transformer = Pipeline(steps=[
    ('missing_num', SimpleImputer(strategy='most_frequent')),
    ('ce', ce.OrdinalEncoder(mapping=clarity_map))
])

fluor_map = [{
    'col': 'fluor_intensity',
    'mapping': {
        'Very Slight': 4,  
        'Faint': 3,
        'Medium': 2,
        'Strong': 1,
        'Very Strong': 0   
    }
}]
fluor_intensity_transformer = Pipeline(steps=[
    ('missing_num', SimpleImputer(strategy='most_frequent')),
    ('ce', ce.OrdinalEncoder(mapping=fluor_map))
])

eyeclean_map = [{
    'col': 'eye_clean',
    'mapping': {
        'Yes': 3,
        'Borderline': 2,
        'No': 1,
        'E1': 0
    }
}]
eyeclean_transformer = Pipeline(steps=[
    ('missing_num', SimpleImputer(strategy='most_frequent')),
    ('ce', ce.OrdinalEncoder(mapping=eyeclean_map))
])


CT = ColumnTransformer([
        ("num_transformer", num_transformer, num_features),
        ("meas_transformer", meas_transformer, meas_features),
        ("girdle_transformer", girdle_transformer, girdle_features),
        ("binary_transformer", binary_transformer, b_ordered_features),
        ("color_map", color_transformer, ['color']),
        ("clarity_map", clarity_transformer, ['clarity']),
        ("fluor_intensity_map", fluor_intensity_transformer, ['fluor_intensity']),
        ("eyeclean_map", eyeclean_transformer, ['eye_clean'])
        ]).set_output(transform='pandas')

display(CT)

ct = CT.fit_transform(X)
pd.DataFrame(ct).head().T

,transformers,"[('num_transformer', ...), ('meas_transformer', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'mean'
,fill_value,None


,0,1,2,3,4
num_transformer__size,0.455224,0.634328,0.111940,0.111940,0.126866
num_transformer__depth_percent,0.276190,0.752381,0.371429,0.638095,0.571429
num_transformer__table_percent,0.588235,0.470588,0.529412,0.470588,0.529412
meas_transformer__meas_length,1.464482,1.779426,-0.664536,-0.777915,-0.588949
meas_transformer__meas_width,1.441174,1.791703,-0.661998,-0.749630,-0.611922
meas_transformer__meas_depth,1.051189,2.015683,-0.857705,-0.717050,-0.596488
meas_transformer__meas_length^2,1.435702,1.802844,-0.654409,-0.746574,-0.591891
meas_transformer__meas_length meas_width,1.422738,1.810574,-0.653500,-0.735353,-0.601576
meas_transformer__meas_length meas_depth,1.206072,1.954890,-0.737115,-0.725048,-0.597482
meas_transformer__meas_width^2,1.409558,1.818008,-0.652524,-0.723987,-0.611169


In [11]:
best_pipe = TransformedTargetRegressor(
    regressor = Pipeline(steps=[
        ('preproc', CT),
        ('estimator', KNeighborsRegressor(
            n_neighbors=12,
            p=1,
            weights='distance',
            n_jobs=-1
        ))
    ]),
    transformer = PowerTransformer() 
)
best_model = best_pipe.fit(X, y)

In [12]:
# Блок предсказания с использованием тестового набора

In [13]:
y_predict = best_model.predict(X_test)

In [14]:
# Название вектора предсказанных значений  y_predict полученого на основании тестового набора
y_predict

array([1282.89933666,  585.87314836,  826.97353535, ..., 2306.25277464,
       3006.86819809, 7394.18174642])